In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/eyadmalharthi/tiktok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-34-09-183_textready_analysis.xlsx
/kaggle/input/datasets/eyadmalharthi/tiktok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-06-43-625_textready_analysis.xlsx
/kaggle/input/datasets/eyadmalharthi/tiktok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-17-06-320_textready_cleaned.xlsx
/kaggle/input/datasets/eyadmalharthi/tiktok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-37-52-643_textready_analysis.xlsx
/kaggle/input/datasets/eyadmalharthi/tiktok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-36-16-596_textready_analysis.xlsx
/kaggle/input/datasets/eyadmalharthi/tiktok-data/Tik Tok Datasets 

In [10]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
import joblib

In [9]:
import os
import pandas as pd
from pathlib import Path

DATA_ROOT = "/kaggle/input"

def read_csv_robust(path):
    encodings = ["utf-8", "utf-8-sig", "cp1256", "latin1"]
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except:
            pass
    return pd.read_csv(path, encoding="latin1", engine="python", on_bad_lines="skip")

def read_excel_robust(path):
    return pd.read_excel(path)

analysis_files = []

for root, _, files in os.walk(DATA_ROOT):
    for file in files:
        name_without_ext = os.path.splitext(file.lower())[0]
        if name_without_ext.endswith("analysis"):
            analysis_files.append(os.path.join(root, file))

print("Analysis files found:", len(analysis_files))
for f in analysis_files:
    print(f)

Analysis files found: 296
/kaggle/input/datasets/eyadmalharthi/tiktok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-34-09-183_textready_analysis.xlsx
/kaggle/input/datasets/eyadmalharthi/tiktok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-06-43-625_textready_analysis.xlsx
/kaggle/input/datasets/eyadmalharthi/tiktok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-37-52-643_textready_analysis.xlsx
/kaggle/input/datasets/eyadmalharthi/tiktok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-36-16-596_textready_analysis.xlsx
/kaggle/input/datasets/eyadmalharthi/tiktok-data/Tik Tok Datasets - After Processing/المنطقة الوسطى/القصيم/dataset_tiktok-comments-scraper_2025-11-05_23-21-11-683_textready_analysis.xlsx
/kaggle/input/datasets/eyadmalharthi/ti

In [8]:
TEXT_PRIORITY = [
    "Text_ML",
    "Text_Cleaned",
    "Text_Normalized",
    "Text_TR",
    "Text_Base",
    "Text",
    "text",
    "Text_Orig"
]

LABEL_PRIORITY = [
    "Sentiment",
    "sentiment",
    "Label",
    "label"
]

def pick_first_existing(columns, priority_list):
    cols = set(columns)
    for c in priority_list:
        if c in cols:
            return c
    return None

dfs = []
failed = []

for path in analysis_files:
    try:
        ext = Path(path).suffix.lower()

        if ext == ".csv":
            df = read_csv_robust(path)
        elif ext in [".xlsx", ".xls"]:
            df = read_excel_robust(path)
        else:
            continue

        text_col = pick_first_existing(df.columns, TEXT_PRIORITY)
        label_col = pick_first_existing(df.columns, LABEL_PRIORITY)

        if text_col is None:
            print("No text column found:", path)
            continue

        out = pd.DataFrame()
        out["text"] = df[text_col].astype(str).fillna("").str.strip()

        if label_col is not None:
            out["label"] = df[label_col]
        else:
            out["label"] = pd.NA

        if "Stars" in df.columns:
            out["Stars"] = df["Stars"]
        elif "stars" in df.columns:
            out["Stars"] = df["stars"]
        else:
            out["Stars"] = pd.NA

        out["file_name"] = Path(path).name
        out = out[out["text"].str.len() > 0].copy()

        dfs.append(out)

    except Exception as e:
        failed.append((path, str(e)))

data = pd.concat(dfs, ignore_index=True)

print("Unified shape:", data.shape)
print(data.head())
print("\nMissing labels:", data["label"].isna().sum())
print("\nLabel distribution:")
print(data["label"].value_counts(dropna=False))

Unified shape: (7280, 4)
                                                text     label Stars  \
0  لايغركم التصوير صغيره المزرعه هي حدود البحيره ...   neutral     3   
1  حلوه بس لو يحطون فعاليات اكثر الشاهي والقوارب ...  positive     5   
2  لا_بالبدائع مرة جميله ودخول السيارة ب١٠والجلسا...  positive     5   
3  حلوه بس ناقصها اشياء كثير مافيها كوفي ابدا ولا...  positive     5   
4           مزرعة الحبردي بالبدائع جميلة جدا EMO_POS  positive     5   

                                           file_name  
0  dataset_tiktok-comments-scraper_2025-11-05_23-...  
1  dataset_tiktok-comments-scraper_2025-11-05_23-...  
2  dataset_tiktok-comments-scraper_2025-11-05_23-...  
3  dataset_tiktok-comments-scraper_2025-11-05_23-...  
4  dataset_tiktok-comments-scraper_2025-11-05_23-...  

Missing labels: 0

Label distribution:
label
positive    3871
negative    2159
neutral     1250
Name: count, dtype: int64


In [7]:
def normalize_label(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip().lower()

    mapping = {
        "positive": "positive",
        "pos": "positive",
        "negative": "negative",
        "neg": "negative",
        "neutral": "neutral",
        "neu": "neutral",
        "mixed": "neutral"
    }

    return mapping.get(x, x)

data["label"] = data["label"].apply(normalize_label)

train_df = data.dropna(subset=["label"]).copy()
train_df = train_df[train_df["label"].isin(["positive", "negative", "neutral"])].copy()

print(train_df.shape)
print(train_df["label"].value_counts())

(7280, 4)
label
positive    3871
negative    2159
neutral     1250
Name: count, dtype: int64


In [26]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# =========================
# MULTI-CLASS MODEL WITH LIGHT GridSearchCV
# =========================
X_train_multi, X_val_multi, y_train_multi, y_val_multi = train_test_split(
    train_df["text"],
    train_df["label"],
    test_size=0.1,
    random_state=42,
    stratify=train_df["label"]
)

word_tfidf = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

features = FeatureUnion([
    ("word", word_tfidf),
    ("char", char_tfidf)
])

linear_svc_multi = Pipeline([
    ("features", features),
    ("clf", LinearSVC(random_state=42, max_iter=5000))
])

param_grid_multi = {
    "features__word__ngram_range": [(1, 2), (1, 3)],
    "features__char__ngram_range": [(3, 5), (3, 6)],
    "clf__C": [0.5, 1.0, 2.0],
    "clf__class_weight": [
        "balanced",
        {"negative": 1.0, "neutral": 2.0, "positive": 1.0},
        {"negative": 1.0, "neutral": 2.3, "positive": 1.0}
    ]
}

grid_multi = GridSearchCV(
    estimator=linear_svc_multi,
    param_grid=param_grid_multi,
    scoring="f1_macro",
    cv=2,
    n_jobs=-1,
    verbose=2
)

grid_multi.fit(X_train_multi, y_train_multi)

best_multi_model = grid_multi.best_estimator_
pred_multi = best_multi_model.predict(X_val_multi)

print("Best params (MULTI):", grid_multi.best_params_)
print("Best CV score (MULTI):", grid_multi.best_score_)
print("Validation Accuracy (MULTI):", accuracy_score(y_val_multi, pred_multi))
print()
print(classification_report(y_val_multi, pred_multi, digits=4))
print(confusion_matrix(y_val_multi, pred_multi))

Fitting 2 folds for each of 36 candidates, totalling 72 fits
Best params (MULTI): {'clf__C': 0.5, 'clf__class_weight': 'balanced', 'features__char__ngram_range': (3, 5), 'features__word__ngram_range': (1, 3)}
Best CV score (MULTI): 0.7062064218373891
Validation Accuracy (MULTI): 0.7967032967032966

              precision    recall  f1-score   support

    negative     0.7404    0.8056    0.7716       216
     neutral     0.6729    0.5760    0.6207       125
    positive     0.8653    0.8630    0.8642       387

    accuracy                         0.7967       728
   macro avg     0.7595    0.7482    0.7522       728
weighted avg     0.7952    0.7967    0.7949       728

[[174  17  25]
 [ 26  72  27]
 [ 35  18 334]]


In [27]:
# =========================
# BINARY MODEL WITH LIGHT GridSearchCV
# neutral removed completely
# =========================
binary_df = train_df[train_df["label"].isin(["positive", "negative"])].copy()

X_train_bin, X_val_bin, y_train_bin, y_val_bin = train_test_split(
    binary_df["text"],
    binary_df["label"],
    test_size=0.1,
    random_state=42,
    stratify=binary_df["label"]
)

linear_svc_binary = Pipeline([
    ("features", features),
    ("clf", LinearSVC(random_state=42, max_iter=5000))
])

param_grid_binary = {
    "features__word__ngram_range": [(1, 2), (1, 3)],
    "features__char__ngram_range": [(3, 5), (3, 6)],
    "clf__C": [0.5, 1.0, 2.0],
    "clf__class_weight": [
        "balanced",
        {"negative": 1.2, "positive": 1.0},
        {"negative": 1.4, "positive": 1.0}
    ]
}

grid_bin = GridSearchCV(
    estimator=linear_svc_binary,
    param_grid=param_grid_binary,
    scoring="f1_macro",
    cv=2,
    n_jobs=-1,
    verbose=2
)

grid_bin.fit(X_train_bin, y_train_bin)

best_binary_model = grid_bin.best_estimator_
pred_bin = best_binary_model.predict(X_val_bin)

print("Best params (BINARY):", grid_bin.best_params_)
print("Best CV score (BINARY):", grid_bin.best_score_)
print("Validation Accuracy (BINARY):", accuracy_score(y_val_bin, pred_bin))
print()
print(classification_report(y_val_bin, pred_bin, digits=4))
print(confusion_matrix(y_val_bin, pred_bin))

Fitting 2 folds for each of 36 candidates, totalling 72 fits
[CV] END clf__C=0.5, clf__class_weight=balanced, features__char__ngram_range=(3, 5), features__word__ngram_range=(1, 2); total time=   1.8s
[CV] END clf__C=0.5, clf__class_weight=balanced, features__char__ngram_range=(3, 6), features__word__ngram_range=(1, 2); total time=   2.0s
[CV] END clf__C=0.5, clf__class_weight={'negative': 1.0, 'neutral': 2.0, 'positive': 1.0}, features__char__ngram_range=(3, 5), features__word__ngram_range=(1, 2); total time=   1.8s
[CV] END clf__C=0.5, clf__class_weight={'negative': 1.0, 'neutral': 2.0, 'positive': 1.0}, features__char__ngram_range=(3, 6), features__word__ngram_range=(1, 2); total time=   2.0s
[CV] END clf__C=0.5, clf__class_weight={'negative': 1.0, 'neutral': 2.3, 'positive': 1.0}, features__char__ngram_range=(3, 5), features__word__ngram_range=(1, 2); total time=   1.8s
[CV] END clf__C=0.5, clf__class_weight={'negative': 1.0, 'neutral': 2.3, 'positive': 1.0}, features__char__ngram_

In [28]:
import joblib

# حفظ مودل الثلاث كلاسات
joblib.dump(best_multi_model, "multi_class_svm.pkl")

# حفظ مودل الباينري
joblib.dump(best_binary_model, "binary_svm.pkl")

print("Models saved successfully ✅")

Models saved successfully ✅

[CV] END clf__C=2.0, clf__class_weight={'negative': 1.2, 'positive': 1.0}, features__char__ngram_range=(3, 5), features__word__ngram_range=(1, 3); total time=   1.4s
[CV] END clf__C=2.0, clf__class_weight={'negative': 1.2, 'positive': 1.0}, features__char__ngram_range=(3, 6), features__word__ngram_range=(1, 3); total time=   1.6s
[CV] END clf__C=2.0, clf__class_weight={'negative': 1.4, 'positive': 1.0}, features__char__ngram_range=(3, 6), features__word__ngram_range=(1, 2); total time=   1.5s

[CV] END clf__C=2.0, clf__class_weight={'negative': 1.2, 'positive': 1.0}, features__char__ngram_range=(3, 5), features__word__ngram_range=(1, 2); total time=   1.3s
[CV] END clf__C=2.0, clf__class_weight={'negative': 1.2, 'positive': 1.0}, features__char__ngram_range=(3, 6), features__word__ngram_range=(1, 2); total time=   1.5s
[CV] END clf__C=2.0, clf__class_weight={'negative': 1.4, 'positive': 1.0}, features__char__ngram_range=(3, 5), features__word__ngram_range=(